# Г1: функциональная связность ЭЭГ против REVE v4
Захар Санников, 08.11.2026. Признаки: корреляция Пирсона и wPLI между 19 каналами в 5 диапазонах (1–30 Гц) + относительная мощность; классификатор — логистическая регрессия (L2) и HistGradientBoosting. Обучение — NMT fit, выбор гиперпараметров по ROC AUC на val; тест — NMT eval, Турция, EpiEEG-60. Оценки REVE v4 (60 окон, верхние 20 %) взяты из `eegdino_v2_per_person.csv`.
Данные: kernel_sources `zakharsann/h1-prep-shared`, `zakharsann/h1-epieeg60-prep`; возраст — `mhmdrdwn/nmt-eeg` (Labels.csv).

In [ ]:
import os
for k in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS']: os.environ[k]='1'
SMOKE = False

In [ ]:
"""Общий набор Г1 (NMT + турецкий) — выход блокнота Kaggle zakharsann/h1-prep-shared.

Подключение в kernel-metadata.json: "kernel_sources": ["zakharsann/h1-prep-shared"].
Блокноты моделей копируют этот файл в начало (одна ячейка) и пользуются функциями ниже.

Формат (подробно — в описании блокнота h1_2026-10-26_zs_prep-shared.ipynb):
- h1_shared_meta.csv — по строке на запись: dataset, part, split, id, y, ..., shard200/off200/n200 (и shard256/... если есть);
- h1_shared_info.json — фильтр, множитель масштаба NMT, сколько минут NMT train сохранено, порядок каналов;
- fs200/<имя>.npy — плоские массивы float16: записи (19, n) в C-порядке подряд, запись = arr[off : off + 19*n].
Сигнал в мкВ, 19 каналов в порядке EL, средний референт, 0,1-75 Гц + режекция 50 Гц. Окна НЕ нормированы.
"""
import glob, json, os
from fractions import Fraction
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, scipy.signal as ss

EL = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2", "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]


def find_root(base="/kaggle/input"):
    """Папка общего набора среди подключённых источников."""
    hits = glob.glob(f"{base}/**/h1_shared_meta.csv", recursive=True)
    assert hits, "общий набор не подключён: добавьте zakharsann/h1-prep-shared в kernel_sources"
    return os.path.dirname(hits[0])


def load_meta(root):
    """Таблица записей и описание набора."""
    meta = pd.read_csv(f"{root}/h1_shared_meta.csv")
    info = json.load(open(f"{root}/h1_shared_info.json", encoding="utf-8"))
    return meta, info


_SHARDS = {}
def _shard(root, fs, name):
    k = (root, fs, name)
    if k not in _SHARDS:
        _SHARDS[k] = np.load(f"{root}/fs{fs}/{name}", mmap_mode="r")
    return _SHARDS[k]


def read_record(root, row, fs=200):
    """Одна запись: (19, n) float16, мкВ. row — строка meta (Series или dict).
    Через mmap — для единичных записей; много записей читать через read_all (на /kaggle/input mmap очень медленный)."""
    n, off = int(row[f"n{fs}"]), int(row[f"off{fs}"])
    return np.array(_shard(root, fs, row[f"shard{fs}"])[off:off + 19 * n]).reshape(19, n)


def read_all(root, meta, fs=200, threads=4):
    """Все записи meta в словарь id -> (19, n) float16: каждый файл-кусок читается целиком последовательно, записи режутся в памяти."""
    def load(item):
        name, g = item
        arr = np.load(f"{root}/fs{fs}/{name}")
        return [(i, arr[o:o + 19 * n].reshape(19, n).copy()) for i, o, n in zip(g["id"], g[f"off{fs}"], g[f"n{fs}"])]
    out = {}
    with ThreadPoolExecutor(threads) as ex:
        for part in ex.map(load, list(meta.groupby(f"shard{fs}"))):
            out.update(part)
    return {i: out[i] for i in meta["id"]}


def windows(X, L, S=None):
    """Окна длиной L с шагом S (по умолчанию S = L, подряд): (k, 19, L)."""
    S = S or L
    k = (X.shape[-1] - L) // S + 1 if X.shape[-1] >= L else 0
    if k <= 0:
        return np.zeros((0,) + X.shape[:-1] + (L,), X.dtype)
    return np.lib.stride_tricks.sliding_window_view(X, L, axis=-1)[..., ::S, :][..., :k, :].swapaxes(0, -2).copy()


def random_windows(X, L, K, rng):
    """K окон длиной L со случайным началом (любой отсчёт) — свежие окна на каждой эпохе: (K, 19, L)."""
    n = X.shape[-1] - L + 1
    if n <= 0:
        return np.zeros((0,) + X.shape[:-1] + (L,), X.dtype)
    st = np.sort(rng.integers(0, n, K))
    return np.stack([X[..., s:s + L] for s in st])


def resample(X, fs_from, fs_to):
    """Пересчёт частоты (полифазный, как scipy.signal.resample_poly) по последней оси, float32 на выходе."""
    if fs_from == fs_to:
        return X
    fr = Fraction(int(fs_to), int(fs_from))
    return ss.resample_poly(np.asarray(X, np.float32), fr.numerator, fr.denominator, axis=-1).astype(np.float32)


def highpass(X, f, fs=200, order=4):
    """Дофильтровка сверху общей нижней границы 0,1 Гц (например, 0,3 Гц как у CBraMod): по всей записи, не по окну."""
    sos = ss.butter(order, f, btype="highpass", fs=fs, output="sos")
    return ss.sosfiltfilt(sos, np.asarray(X, np.float64), axis=-1).astype(np.float32)

In [ ]:
import time, warnings, io
from multiprocessing import Pool
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, roc_curve
from scipy import stats
warnings.filterwarnings("ignore")
T00 = time.time()
OUT = "/kaggle/working"
REVE_CSV = """dataset,id,y,reve4_ens_prob
nmt_val,0000018.edf,0,0.8889699
nmt_val,0000032.edf,0,0.66765386
nmt_val,0000033.edf,0,0.47852144
nmt_val,0000034.edf,0,0.65715694
nmt_val,0000038.edf,0,0.12400975
nmt_val,0000048.edf,0,0.41886595
nmt_val,0000058.edf,0,0.54062223
nmt_val,0000081.edf,0,0.73816603
nmt_val,0000090.edf,1,0.95586747
nmt_val,0000095.edf,0,0.25069958
nmt_val,0000101.edf,0,0.5414803
nmt_val,0000102.edf,0,0.5734046
nmt_val,0000114.edf,0,0.336629
nmt_val,0000115.edf,1,0.92546564
nmt_val,0000119.edf,0,0.8563028
nmt_val,0000120.edf,0,0.09765508
nmt_val,0000126.edf,0,0.72138697
nmt_val,0000129.edf,0,0.36645135
nmt_val,0000133.edf,0,0.7704924
nmt_val,0000134.edf,0,0.7651341
nmt_val,0000147.edf,0,0.7396476
nmt_val,0000152.edf,0,0.6836341
nmt_val,0000154.edf,1,0.9304743
nmt_val,0000160.edf,0,0.5909324
nmt_val,0000162.edf,1,0.7151098
nmt_val,0000165.edf,0,0.8642904
nmt_val,0000171.edf,0,0.45013687
nmt_val,0000173.edf,0,0.8747831
nmt_val,0000183.edf,0,0.77987814
nmt_val,0000188.edf,1,0.9453163
nmt_val,0000195.edf,0,0.7541242
nmt_val,0000197.edf,0,0.65489316
nmt_val,0000222.edf,1,0.73018104
nmt_val,0000227.edf,0,0.75552684
nmt_val,0000230.edf,0,0.56186515
nmt_val,0000235.edf,0,0.7504863
nmt_val,0000241.edf,1,0.94631463
nmt_val,0000244.edf,0,0.779336
nmt_val,0000248.edf,1,0.9696557
nmt_val,0000271.edf,0,0.34342992
nmt_val,0000273.edf,0,0.6599687
nmt_val,0000277.edf,0,0.8925871
nmt_val,0000281.edf,0,0.8510384
nmt_val,0000284.edf,0,0.6123478
nmt_val,0000289.edf,0,0.6427602
nmt_val,0000294.edf,0,0.36150023
nmt_val,0000314.edf,0,0.35852993
nmt_val,0000327.edf,0,0.49472275
nmt_val,0000341.edf,0,0.8672072
nmt_val,0000342.edf,0,0.694752
nmt_val,0000362.edf,1,0.9238994
nmt_val,0000364.edf,0,0.31156352
nmt_val,0000365.edf,0,0.15107651
nmt_val,0000366.edf,0,0.49843338
nmt_val,0000370.edf,0,0.9464474
nmt_val,0000379.edf,0,0.6935799
nmt_val,0000380.edf,0,0.8043267
nmt_val,0000385.edf,1,0.8705399
nmt_val,0000386.edf,0,0.8873778
nmt_val,0000392.edf,0,0.52140707
nmt_val,0000393.edf,0,0.7681553
nmt_val,0000408.edf,1,0.99333525
nmt_val,0000409.edf,0,0.80065185
nmt_val,0000416.edf,0,0.3567346
nmt_val,0000419.edf,1,0.8805265
nmt_val,0000430.edf,0,0.0250555
nmt_val,0000431.edf,0,0.21077959
nmt_val,0000446.edf,0,0.75046045
nmt_val,0000453.edf,0,0.2507774
nmt_val,0000483.edf,0,0.7428685
nmt_val,0000486.edf,0,0.6834678
nmt_val,0000490.edf,0,0.64849466
nmt_val,0000495.edf,0,0.7268875
nmt_val,0000498.edf,1,0.78615624
nmt_val,0000500.edf,1,0.9565694
nmt_val,0000508.edf,0,0.29617238
nmt_val,0000514.edf,0,0.65991306
nmt_val,0000515.edf,0,0.6222742
nmt_val,0000525.edf,0,0.06763711
nmt_val,0000562.edf,0,0.39813504
nmt_val,0000569.edf,0,0.73596865
nmt_val,0000570.edf,0,0.3431469
nmt_val,0000587.edf,0,0.30780876
nmt_val,0000591.edf,0,0.7827856
nmt_val,0000593.edf,0,0.62715167
nmt_val,0000615.edf,0,0.311766
nmt_val,0000637.edf,0,0.85485935
nmt_val,0000664.edf,0,0.47366714
nmt_val,0000682.edf,0,0.50214416
nmt_val,0000694.edf,0,0.13159852
nmt_val,0000700.edf,0,0.48423848
nmt_val,0000701.edf,0,0.21843489
nmt_val,0000704.edf,0,0.6129946
nmt_val,0000708.edf,0,0.28812298
nmt_val,0000713.edf,0,0.78597754
nmt_val,0000718.edf,0,0.2173431
nmt_val,0000719.edf,0,0.6149774
nmt_val,0000721.edf,0,0.28722453
nmt_val,0000754.edf,0,0.69251686
nmt_val,0000757.edf,0,0.2981305
nmt_val,0000761.edf,0,0.6183438
nmt_val,0000762.edf,0,0.27531394
nmt_val,0000763.edf,0,0.30816934
nmt_val,0000770.edf,0,0.41595137
nmt_val,0000774.edf,0,0.044344682
nmt_val,0000790.edf,0,0.6116366
nmt_val,0000801.edf,0,0.70824796
nmt_val,0000806.edf,0,0.5749836
nmt_val,0000808.edf,0,0.39815512
nmt_val,0000833.edf,0,0.599378
nmt_val,0000837.edf,0,0.6216883
nmt_val,0000852.edf,0,0.87948066
nmt_val,0000854.edf,0,0.87160796
nmt_val,0000864.edf,0,0.30741808
nmt_val,0000879.edf,0,0.22364068
nmt_val,0000881.edf,0,0.9385665
nmt_val,0000887.edf,0,0.78031635
nmt_val,0000891.edf,0,0.860239
nmt_val,0000903.edf,0,0.62626594
nmt_val,0000915.edf,0,0.77704525
nmt_val,0000924.edf,0,0.52235764
nmt_val,0000927.edf,1,0.94882435
nmt_val,0000929.edf,0,0.67203116
nmt_val,0000933.edf,0,0.5991559
nmt_val,0000935.edf,0,0.66851777
nmt_val,0000939.edf,0,0.33000568
nmt_val,0000940.edf,1,0.83178854
nmt_val,0000951.edf,0,0.14510126
nmt_val,0000957.edf,0,0.8913348
nmt_val,0000958.edf,0,0.50459015
nmt_val,0000965.edf,0,0.489985
nmt_val,0000972.edf,0,0.5070129
nmt_val,0000976.edf,0,0.7688554
nmt_val,0000979.edf,0,0.48009935
nmt_val,0000980.edf,0,0.67224455
nmt_val,0000981.edf,0,0.31939253
nmt_val,0000984.edf,0,0.4850073
nmt_val,0000996.edf,0,0.9075405
nmt_val,0001003.edf,0,0.16950147
nmt_val,0001008.edf,0,0.8393119
nmt_val,0001012.edf,0,0.8791086
nmt_val,0001013.edf,0,0.13496709
nmt_val,0001019.edf,0,0.77728313
nmt_val,0001024.edf,0,0.66299033
nmt_val,0001030.edf,0,0.81307286
nmt_val,0001048.edf,0,0.26412484
nmt_val,0001059.edf,0,0.9013734
nmt_val,0001061.edf,0,0.48237216
nmt_val,0001066.edf,0,0.49709687
nmt_val,0001068.edf,0,0.820659
nmt_val,0001082.edf,1,0.8779475
nmt_val,0001088.edf,0,0.13594392
nmt_val,0001093.edf,0,0.99112755
nmt_val,0001100.edf,0,0.68248606
nmt_val,0001106.edf,0,0.8277213
nmt_val,0001110.edf,0,0.7352924
nmt_val,0001112.edf,0,0.7133572
nmt_val,0001113.edf,0,0.7686581
nmt_val,0001119.edf,1,0.9502582
nmt_val,0001125.edf,1,0.8491724
nmt_val,0001139.edf,0,0.8530237
nmt_val,0001141.edf,0,0.8032196
nmt_val,0001142.edf,0,0.5052205
nmt_val,0001151.edf,0,0.32273284
nmt_val,0001155.edf,0,0.84871215
nmt_val,0001165.edf,0,0.5810837
nmt_val,0001168.edf,0,0.54014206
nmt_val,0001172.edf,0,0.43618765
nmt_val,0001194.edf,0,0.62127924
nmt_val,0001201.edf,0,0.6309397
nmt_val,0001205.edf,0,0.5981744
nmt_val,0001215.edf,0,0.87410635
nmt_val,0001225.edf,0,0.82715845
nmt_val,0001226.edf,0,0.6796787
nmt_val,0001229.edf,0,0.75180036
nmt_val,0001231.edf,0,0.60403264
nmt_val,0001238.edf,1,0.8481447
nmt_val,0001241.edf,1,0.9946173
nmt_val,0001258.edf,1,0.9512473
nmt_val,0001268.edf,0,0.25550196
nmt_val,0001271.edf,1,0.8459231
nmt_val,0001273.edf,0,0.7115516
nmt_val,0001287.edf,0,0.8215518
nmt_val,0001291.edf,0,0.7021474
nmt_val,0001307.edf,0,0.57853055
nmt_val,0001309.edf,0,0.7679135
nmt_val,0001319.edf,0,0.6798744
nmt_val,0001340.edf,0,0.8918052
nmt_val,0001345.edf,0,0.8668175
nmt_val,0001346.edf,0,0.8747507
nmt_val,0001349.edf,0,0.8361306
nmt_val,0001352.edf,1,0.8484288
nmt_val,0001357.edf,0,0.30028024
nmt_val,0001365.edf,0,0.60415363
nmt_val,0001371.edf,0,0.0706888
nmt_val,0001374.edf,0,0.5599869
nmt_val,0001386.edf,0,0.58767253
nmt_val,0001397.edf,0,0.804626
nmt_val,0001400.edf,0,0.5705943
nmt_val,0001418.edf,0,0.6121529
nmt_val,0001440.edf,0,0.60165936
nmt_val,0001448.edf,0,0.74138236
nmt_val,0001452.edf,0,0.6356018
nmt_val,0001465.edf,0,0.7335221
nmt_val,0001467.edf,0,0.62772757
nmt_val,0001469.edf,0,0.8702607
nmt_val,0001477.edf,0,0.18580526
nmt_val,0001482.edf,0,0.6896698
nmt_val,0001487.edf,0,0.33059585
nmt_val,0001488.edf,0,0.4018401
nmt_val,0001492.edf,0,0.42482877
nmt_val,0001519.edf,0,0.5371467
nmt_val,0001523.edf,0,0.5832165
nmt_val,0001526.edf,0,0.8215332
nmt_val,0001528.edf,0,0.43971172
nmt_val,0001529.edf,0,0.40533245
nmt_val,0001530.edf,0,0.23910858
nmt_val,0001542.edf,0,0.6849616
nmt_val,0001547.edf,0,0.4247813
nmt_val,0001575.edf,0,0.74160033
nmt_val,0001585.edf,0,0.8938809
nmt_val,0001587.edf,0,0.51374346
nmt_val,0001588.edf,0,0.8542113
nmt_val,0001618.edf,0,0.78705055
nmt_val,0001632.edf,0,0.5176316
nmt_val,0001637.edf,0,0.53205603
nmt_val,0001643.edf,0,0.53865194
nmt_val,0001652.edf,1,0.678449
nmt_val,0001658.edf,1,0.8999815
nmt_val,0001659.edf,0,0.7726479
nmt_val,0001668.edf,0,0.6225633
nmt_val,0001673.edf,0,0.7084782
nmt_val,0001686.edf,1,0.771713
nmt_val,0001692.edf,0,0.9539707
nmt_val,0001708.edf,1,0.911174
nmt_val,0001715.edf,0,0.24988401
nmt_val,0001720.edf,0,0.7335383
nmt_val,0001722.edf,0,0.44127858
nmt_val,0001746.edf,1,0.97918254
nmt_val,0001747.edf,0,0.63263845
nmt_val,0001757.edf,0,0.75689906
nmt_val,0001769.edf,0,0.7755241
nmt_val,0001781.edf,0,0.37899604
nmt_val,0001783.edf,0,0.5794967
nmt_val,0001793.edf,0,0.5175348
nmt_val,0001795.edf,0,0.7298084
nmt_val,0001797.edf,0,0.8900506
nmt_val,0001812.edf,0,0.91786504
nmt_val,0001813.edf,0,0.63201547
nmt_val,0001818.edf,1,0.9797399
nmt_val,0001824.edf,0,0.7466612
nmt_val,0001828.edf,0,0.12839921
nmt_val,0001839.edf,0,0.46030846
nmt_val,0001848.edf,0,0.7459192
nmt_val,0001853.edf,0,0.78337955
nmt_val,0001859.edf,0,0.7949473
nmt_val,0001863.edf,0,0.8483587
nmt_val,0001874.edf,0,0.49363413
nmt_val,0001876.edf,0,0.5132384
nmt_val,0001879.edf,0,0.56890637
nmt_val,0001889.edf,0,0.30726227
nmt_val,0001908.edf,0,0.64151573
nmt_val,0001910.edf,0,0.56729096
nmt_val,0001930.edf,0,0.7674213
nmt_val,0001935.edf,0,0.45378962
nmt_val,0001938.edf,0,0.8370428
nmt_val,0001952.edf,0,0.6734821
nmt_val,0001965.edf,0,0.7853604
nmt_val,0001970.edf,0,0.30821565
nmt_val,0001979.edf,0,0.26547655
nmt_val,0001981.edf,0,0.5914174
nmt_val,0001983.edf,0,0.8571727
nmt_val,0001994.edf,0,0.4616455
nmt_val,0002004.edf,0,0.835514
nmt_val,0002008.edf,1,0.7459442
nmt_val,0002010.edf,1,0.86382526
nmt_val,0002013.edf,0,0.93342113
nmt_val,0002018.edf,0,0.73373765
nmt_val,0002027.edf,0,0.75218886
nmt_val,0002034.edf,0,0.52219653
nmt_val,0002039.edf,0,0.78332776
nmt_val,0002048.edf,0,0.33732367
nmt_val,0002055.edf,0,0.73647577
nmt_val,0002061.edf,1,0.75335354
nmt_val,0002062.edf,1,0.83413774
nmt_val,0002066.edf,0,0.5726111
nmt_val,0002067.edf,0,0.8007989
nmt_val,0002069.edf,0,0.5137821
nmt_val,0002071.edf,0,0.6884715
nmt_val,0002079.edf,1,0.76339936
nmt_val,0002082.edf,1,0.9869859
nmt_val,0002084.edf,0,0.7951632
nmt_val,0002094.edf,1,0.6795492
nmt_val,0002100.edf,1,0.9124933
nmt_val,0002127.edf,0,0.5315995
nmt_val,0002134.edf,1,0.95432615
nmt_val,0002136.edf,1,0.92188287
nmt_val,0002144.edf,1,0.9661224
nmt_val,0002147.edf,0,0.21170382
nmt_val,0002150.edf,0,0.11066404
nmt_val,0002161.edf,1,0.83513784
nmt_val,0002167.edf,0,0.7822773
nmt_val,0002177.edf,1,0.85377955
nmt_val,0002179.edf,0,0.066256166
nmt_val,0002182.edf,0,0.62506443
nmt_val,0002189.edf,0,0.5290721
nmt_val,0002191.edf,0,0.45351627
nmt_val,0002193.edf,0,0.87398213
nmt_val,0002199.edf,0,0.03754154
nmt_val,0002200.edf,1,0.9616657
nmt_val,0002201.edf,1,0.98342985
nmt_val,0002204.edf,1,0.97292835
nmt_val,0002206.edf,0,0.48720345
nmt_val,0002208.edf,0,0.4248345
nmt_val,0002212.edf,0,0.97510105
nmt_val,0002223.edf,0,0.7849801
nmt_val,0002241.edf,0,0.48951355
nmt_val,0002250.edf,0,0.6995148
nmt_val,0002251.edf,0,0.8446176
nmt_val,0002261.edf,1,0.9269181
nmt_val,0002270.edf,0,0.7913132
nmt_val,0002280.edf,0,0.6974697
nmt_val,0002307.edf,0,0.69251996
nmt_val,0002309.edf,0,0.51051646
nmt_val,0002317.edf,0,0.86516994
nmt_val,0002328.edf,0,0.08987982
nmt_val,0002329.edf,1,0.8990478
nmt_val,0002333.edf,0,0.60361487
nmt_val,0002347.edf,0,0.63697666
nmt_val,0002349.edf,0,0.35938016
nmt_val,0002358.edf,1,0.9206411
nmt_val,0002371.edf,0,0.7960339
nmt_val,0002376.edf,0,0.72423315
nmt_val,0002392.edf,0,0.39167786
nmt_val,0002409.edf,0,0.7381244
nmt_eval,0000024.edf,0,0.86470896
nmt_eval,0000036.edf,1,0.9903267
nmt_eval,0000044.edf,0,0.9445097
nmt_eval,0000046.edf,1,0.7564538
nmt_eval,0000074.edf,0,0.55375725
nmt_eval,0000076.edf,1,0.8922431
nmt_eval,0000077.edf,0,0.13912593
nmt_eval,0000080.edf,1,0.8769627
nmt_eval,0000093.edf,1,0.63035905
nmt_eval,0000116.edf,1,0.8925647
nmt_eval,0000117.edf,0,0.12162701
nmt_eval,0000123.edf,1,0.8967743
nmt_eval,0000124.edf,0,0.82239336
nmt_eval,0000156.edf,0,0.6730358
nmt_eval,0000194.edf,0,0.79918814
nmt_eval,0000203.edf,1,0.98940665
nmt_eval,0000204.edf,0,0.7465932
nmt_eval,0000206.edf,1,0.9553998
nmt_eval,0000207.edf,0,0.6803496
nmt_eval,0000214.edf,0,0.80777884
nmt_eval,0000246.edf,1,0.999504
nmt_eval,0000254.edf,0,0.61653537
nmt_eval,0000256.edf,1,0.966719
nmt_eval,0000274.edf,0,0.88143754
nmt_eval,0000276.edf,0,0.23994036
nmt_eval,0000283.edf,1,0.7630143
nmt_eval,0000286.edf,1,0.8389706
nmt_eval,0000316.edf,1,0.71765417
nmt_eval,0000336.edf,0,0.6511887
nmt_eval,0000346.edf,1,0.69102985
nmt_eval,0000363.edf,1,0.8739157
nmt_eval,0000396.edf,0,0.7883292
nmt_eval,0000406.edf,1,0.98246545
nmt_eval,0000426.edf,1,0.89603704
nmt_eval,0000436.edf,0,0.41594696
nmt_eval,0000454.edf,0,0.844346
nmt_eval,0000476.edf,1,0.9197741
nmt_eval,0000496.edf,1,0.92351717
nmt_eval,0000504.edf,0,0.55878234
nmt_eval,0000506.edf,1,0.7587526
nmt_eval,0000513.edf,1,0.82889795
nmt_eval,0000526.edf,0,0.23703636
nmt_eval,0000537.edf,0,0.8098302
nmt_eval,0000544.edf,0,0.79542494
nmt_eval,0000546.edf,1,0.916858
nmt_eval,0000566.edf,0,0.26508877
nmt_eval,0000573.edf,1,0.9879485
nmt_eval,0000583.edf,1,0.9358384
nmt_eval,0000584.edf,0,0.7511838
nmt_eval,0000596.edf,0,0.5270106
nmt_eval,0000614.edf,0,0.7507306
nmt_eval,0000616.edf,0,0.3007747
nmt_eval,0000617.edf,0,0.06619581
nmt_eval,0000624.edf,0,0.05462797
nmt_eval,0000643.edf,1,0.9755776
nmt_eval,0000666.edf,0,0.7162679
nmt_eval,0000676.edf,0,0.6343238
nmt_eval,0000714.edf,0,0.16717505
nmt_eval,0000744.edf,0,0.7431174
nmt_eval,0000746.edf,1,0.8978126
nmt_eval,0000783.edf,1,0.8438437
nmt_eval,0000784.edf,0,0.7858879
nmt_eval,0000786.edf,1,0.8467467
nmt_eval,0000816.edf,1,0.90185666
nmt_eval,0000823.edf,1,0.852354
nmt_eval,0000824.edf,0,0.56308943
nmt_eval,0000826.edf,1,0.9937245
nmt_eval,0000838.edf,1,0.76453394
nmt_eval,0000844.edf,0,0.5751936
nmt_eval,0000866.edf,1,0.97966313
nmt_eval,0000868.edf,1,0.9863835
nmt_eval,0000886.edf,1,0.91841274
nmt_eval,0000894.edf,0,0.33842775
nmt_eval,0000896.edf,1,0.31520998
nmt_eval,0000906.edf,1,0.8012046
nmt_eval,0000914.edf,0,0.40118787
nmt_eval,0000916.edf,1,0.9821491
nmt_eval,0000926.edf,1,0.9689352
nmt_eval,0000934.edf,0,0.46383262
nmt_eval,0000947.edf,0,0.48668125
nmt_eval,0000986.edf,0,0.40191114
nmt_eval,0000993.edf,1,0.75916797
nmt_eval,0001016.edf,1,0.9651208
nmt_eval,0001033.edf,1,0.83020264
nmt_eval,0001056.edf,0,0.42917717
nmt_eval,0001057.edf,0,0.6028423
nmt_eval,0001096.edf,0,0.6646263
nmt_eval,0001114.edf,0,0.35813245
nmt_eval,0001146.edf,0,0.41063932
nmt_eval,0001153.edf,1,0.86361855
nmt_eval,0001156.edf,1,0.8987277
nmt_eval,0001161.edf,1,0.9664491
nmt_eval,0001164.edf,0,0.8103779
nmt_eval,0001173.edf,1,0.8459849
nmt_eval,0001183.edf,1,0.90808916
nmt_eval,0001196.edf,0,0.5100545
nmt_eval,0001204.edf,0,0.5512355
nmt_eval,0001216.edf,1,0.95913
nmt_eval,0001234.edf,0,0.5497076
nmt_eval,0001264.edf,0,0.6824576
nmt_eval,0001266.edf,1,0.7969392
nmt_eval,0001267.edf,0,0.55967265
nmt_eval,0001293.edf,1,0.994124
nmt_eval,0001302.edf,1,0.9971002
nmt_eval,0001306.edf,1,0.85451174
nmt_eval,0001331.edf,1,0.95533055
nmt_eval,0001333.edf,1,0.97000337
nmt_eval,0001347.edf,0,0.8963229
nmt_eval,0001353.edf,1,0.79330987
nmt_eval,0001356.edf,0,0.55947226
nmt_eval,0001376.edf,0,0.09642372
nmt_eval,0001383.edf,1,0.78752714
nmt_eval,0001406.edf,1,0.94407207
nmt_eval,0001416.edf,0,0.5715721
nmt_eval,0001433.edf,1,0.93390673
nmt_eval,0001436.edf,1,0.75982183
nmt_eval,0001447.edf,0,0.28514934
nmt_eval,0001454.edf,0,0.6643572
nmt_eval,0001456.edf,0,0.8571694
nmt_eval,0001466.edf,1,0.9904596
nmt_eval,0001486.edf,1,0.99507946
nmt_eval,0001496.edf,0,0.43922368
nmt_eval,0001513.edf,1,0.9838302
nmt_eval,0001534.edf,0,0.57070625
nmt_eval,0001536.edf,0,0.22231202
nmt_eval,0001566.edf,0,0.37217924
nmt_eval,0001573.edf,1,0.8771232
nmt_eval,0001574.edf,0,0.38861108
nmt_eval,0001583.edf,1,0.59641933
nmt_eval,0001586.edf,1,0.98864216
nmt_eval,0001593.edf,1,0.8052826
nmt_eval,0001594.edf,0,0.76427144
nmt_eval,0001616.edf,1,0.9805514
nmt_eval,0001653.edf,1,0.8478872
nmt_eval,0001683.edf,1,0.8806441
nmt_eval,0001684.edf,0,0.65792745
nmt_eval,0001703.edf,1,0.65440273
nmt_eval,0001706.edf,0,0.7092781
nmt_eval,0001713.edf,1,0.8849136
nmt_eval,0001724.edf,0,0.7834647
nmt_eval,0001776.edf,0,0.5998208
nmt_eval,0001794.edf,0,0.61440843
nmt_eval,0001826.edf,0,0.16690041
nmt_eval,0001833.edf,1,0.89078504
nmt_eval,0001843.edf,1,0.9736707
nmt_eval,0001854.edf,0,0.12997693
nmt_eval,0001856.edf,0,0.79421145
nmt_eval,0001887.edf,1,0.8592288
nmt_eval,0001893.edf,1,0.8590772
nmt_eval,0001894.edf,0,0.4899397
nmt_eval,0001896.edf,0,0.2731642
nmt_eval,0001953.edf,1,0.9399454
nmt_eval,0001966.edf,0,0.5917771
nmt_eval,0001967.edf,0,0.19429208
nmt_eval,0002016.edf,0,0.59981036
nmt_eval,0002047.edf,0,0.529234
nmt_eval,0002053.edf,1,0.8853333
nmt_eval,0002063.edf,1,0.7752042
nmt_eval,0002064.edf,0,0.7457803
nmt_eval,0002091.edf,1,0.9463835
nmt_eval,0002106.edf,0,0.83280176
nmt_eval,0002114.edf,0,0.20330854
nmt_eval,0002143.edf,1,0.9054043
nmt_eval,0002153.edf,1,0.7940859
nmt_eval,0002164.edf,0,0.5471664
nmt_eval,0002166.edf,0,0.6237094
nmt_eval,0002181.edf,1,0.73083514
nmt_eval,0002196.edf,0,0.88648933
nmt_eval,0002203.edf,1,0.9009507
nmt_eval,0002244.edf,0,0.37022424
nmt_eval,0002249.edf,1,0.9178416
nmt_eval,0002266.edf,1,0.94944865
nmt_eval,0002276.edf,0,0.82115316
nmt_eval,0002303.edf,1,0.9619083
nmt_eval,0002304.edf,0,0.33677292
nmt_eval,0002316.edf,0,0.8827255
nmt_eval,0002339.edf,1,0.748096
nmt_eval,0002344.edf,0,0.8862583
nmt_eval,0002366.edf,0,0.6463881
nmt_eval,0002373.edf,1,0.99447346
nmt_eval,0002375.edf,0,0.38043734
nmt_eval,0002380.edf,1,0.9285863
nmt_eval,0002384.edf,0,0.5519306
nmt_eval,0002406.edf,0,0.8554458
nmt_eval,0002407.edf,0,0.7306421
turkey,1_epilepsy (1).mat,1,0.66265243
turkey,1_epilepsy (10).mat,1,0.8594267
turkey,1_epilepsy (11).mat,1,0.421748
turkey,1_epilepsy (12).mat,1,0.2647535
turkey,1_epilepsy (13).mat,1,0.36672163
turkey,1_epilepsy (14).mat,1,0.47264275
turkey,1_epilepsy (15).mat,1,0.58585
turkey,1_epilepsy (16).mat,1,0.7872084
turkey,1_epilepsy (17).mat,1,0.77949643
turkey,1_epilepsy (18).mat,1,0.864505
turkey,1_epilepsy (19).mat,1,0.43730965
turkey,1_epilepsy (2).mat,1,0.5914646
turkey,1_epilepsy (20).mat,1,0.60875946
turkey,1_epilepsy (21).mat,1,0.5804963
turkey,1_epilepsy (22).mat,1,0.30493298
turkey,1_epilepsy (23).mat,1,0.50077087
turkey,1_epilepsy (24).mat,1,0.31154016
turkey,1_epilepsy (25).mat,1,0.5312479
turkey,1_epilepsy (26).mat,1,0.369432
turkey,1_epilepsy (27).mat,1,0.46863866
turkey,1_epilepsy (28).mat,1,0.3522689
turkey,1_epilepsy (29).mat,1,0.6652186
turkey,1_epilepsy (3).mat,1,0.5779021
turkey,1_epilepsy (30).mat,1,0.38765025
turkey,1_epilepsy (31).mat,1,0.6192169
turkey,1_epilepsy (32).mat,1,0.36185622
turkey,1_epilepsy (33).mat,1,0.34311748
turkey,1_epilepsy (34).mat,1,0.76709086
turkey,1_epilepsy (35).mat,1,0.45284605
turkey,1_epilepsy (36).mat,1,0.63349336
turkey,1_epilepsy (37).mat,1,0.34447518
turkey,1_epilepsy (38).mat,1,0.33937573
turkey,1_epilepsy (39).mat,1,0.3925661
turkey,1_epilepsy (4).mat,1,0.65982586
turkey,1_epilepsy (40).mat,1,0.4378779
turkey,1_epilepsy (41).mat,1,0.5144961
turkey,1_epilepsy (42).mat,1,0.6841266
turkey,1_epilepsy (43).mat,1,0.31766632
turkey,1_epilepsy (44).mat,1,0.29646745
turkey,1_epilepsy (45).mat,1,0.32198927
turkey,1_epilepsy (46).mat,1,0.678008
turkey,1_epilepsy (47).mat,1,0.6800507
turkey,1_epilepsy (48).mat,1,0.44999933
turkey,1_epilepsy (49).mat,1,0.6536617
turkey,1_epilepsy (5).mat,1,0.37725726
turkey,1_epilepsy (50).mat,1,0.6999581
turkey,1_epilepsy (6).mat,1,0.31831053
turkey,1_epilepsy (7).mat,1,0.1641819
turkey,1_epilepsy (8).mat,1,0.4322541
turkey,1_epilepsy (9).mat,1,0.38666037
turkey,2_Normal (1).mat,0,0.21142383
turkey,2_Normal (10).mat,0,0.23451746
turkey,2_Normal (11).mat,0,0.24566288
turkey,2_Normal (12).mat,0,0.36369172
turkey,2_Normal (13).mat,0,0.24871023
turkey,2_Normal (14).mat,0,0.18227102
turkey,2_Normal (15).mat,0,0.30687878
turkey,2_Normal (16).mat,0,0.30403718
turkey,2_Normal (17).mat,0,0.20567335
turkey,2_Normal (18).mat,0,0.31501603
turkey,2_Normal (19).mat,0,0.2269408
turkey,2_Normal (2).mat,0,0.36900714
turkey,2_Normal (20).mat,0,0.3049777
turkey,2_Normal (21).mat,0,0.28192472
turkey,2_Normal (22).mat,0,0.28192472
turkey,2_Normal (23).mat,0,0.20777889
turkey,2_Normal (24).mat,0,0.33736
turkey,2_Normal (25).mat,0,0.20347065
turkey,2_Normal (26).mat,0,0.34629247
turkey,2_Normal (27).mat,0,0.26305315
turkey,2_Normal (28).mat,0,0.19488071
turkey,2_Normal (29).mat,0,0.19656914
turkey,2_Normal (3).mat,0,0.2210164
turkey,2_Normal (30).mat,0,0.2562298
turkey,2_Normal (31).mat,0,0.3571434
turkey,2_Normal (32).mat,0,0.16668667
turkey,2_Normal (33).mat,0,0.21848463
turkey,2_Normal (34).mat,0,0.1799084
turkey,2_Normal (35).mat,0,0.37747017
turkey,2_Normal (36).mat,0,0.23819292
turkey,2_Normal (37).mat,0,0.18871969
turkey,2_Normal (38).mat,0,0.25909954
turkey,2_Normal (39).mat,0,0.3089141
turkey,2_Normal (4).mat,0,0.2318517
turkey,2_Normal (40).mat,0,0.26933387
turkey,2_Normal (41).mat,0,0.25656316
turkey,2_Normal (42).mat,0,0.33756557
turkey,2_Normal (43).mat,0,0.33445403
turkey,2_Normal (44).mat,0,0.21624207
turkey,2_Normal (45).mat,0,0.20367168
turkey,2_Normal (46).mat,0,0.13969283
turkey,2_Normal (47).mat,0,0.23932995
turkey,2_Normal (48).mat,0,0.26568154
turkey,2_Normal (49).mat,0,0.17123652
turkey,2_Normal (5).mat,0,0.6266572
turkey,2_Normal (50).mat,0,0.22281621
turkey,2_Normal (51).mat,0,0.1044808
turkey,2_Normal (52).mat,0,0.2661848
turkey,2_Normal (53).mat,0,0.2661848
turkey,2_Normal (54).mat,0,0.42629978
turkey,2_Normal (55).mat,0,0.15734565
turkey,2_Normal (56).mat,0,0.26237786
turkey,2_Normal (57).mat,0,0.2577647
turkey,2_Normal (58).mat,0,0.2915949
turkey,2_Normal (59).mat,0,0.23378503
turkey,2_Normal (6).mat,0,0.11230835
turkey,2_Normal (60).mat,0,0.2795787
turkey,2_Normal (61).mat,0,0.25163
turkey,2_Normal (62).mat,0,0.3456045
turkey,2_Normal (63).mat,0,0.28604665
turkey,2_Normal (64).mat,0,0.18026368
turkey,2_Normal (65).mat,0,0.39425245
turkey,2_Normal (66).mat,0,0.25005636
turkey,2_Normal (67).mat,0,0.23132837
turkey,2_Normal (68).mat,0,0.3153654
turkey,2_Normal (69).mat,0,0.2596086
turkey,2_Normal (7).mat,0,0.18098338
turkey,2_Normal (70).mat,0,0.2460643
turkey,2_Normal (71).mat,0,0.32084197
turkey,2_Normal (8).mat,0,0.3427081
turkey,2_Normal (9).mat,0,0.2712494
epieeg60,Abnormal_Patient_31.EEG,1,0.69977427
epieeg60,Abnormal_Patient_32.EEG,1,0.6357491
epieeg60,Abnormal_Patient_33.EEG,1,0.8776829
epieeg60,Abnormal_Patient_34.EEG,1,0.93443227
epieeg60,Abnormal_Patient_35.EEG,1,0.87220424
epieeg60,Abnormal_Patient_36.EEG,1,0.50370115
epieeg60,Abnormal_Patient_37.EEG,1,0.8763538
epieeg60,Abnormal_Patient_38.EEG,1,0.88489014
epieeg60,Abnormal_Patient_39.EEG,1,0.5709393
epieeg60,Abnormal_Patient_40.EEG,1,0.925204
epieeg60,Abnormal_Patient_41.EEG,1,0.88077193
epieeg60,Abnormal_Patient_42.EEG,1,0.3522795
epieeg60,Abnormal_Patient_43.EEG,1,0.9775675
epieeg60,Abnormal_Patient_44.EEG,1,0.57398605
epieeg60,Abnormal_Patient_45.EEG,1,0.8918963
epieeg60,Abnormal_Patient_46.EEG,1,0.87482
epieeg60,Abnormal_Patient_47.EEG,1,0.88745695
epieeg60,Abnormal_Patient_48.EEG,1,0.80183357
epieeg60,Abnormal_Patient_49.EEG,1,0.8946472
epieeg60,Abnormal_Patient_50.EEG,1,0.8849408
epieeg60,Abnormal_Patient_51.EEG,1,0.91529655
epieeg60,Abnormal_Patient_52.EEG,1,0.8142717
epieeg60,Abnormal_Patient_53.EEG,1,0.9176722
epieeg60,Abnormal_Patient_54.EEG,1,0.88045126
epieeg60,Abnormal_Patient_55.EEG,1,0.5689382
epieeg60,Abnormal_Patient_56.EEG,1,0.907469
epieeg60,Abnormal_Patient_57.EEG,1,0.5871131
epieeg60,Abnormal_Patient_58.EEG,1,0.3489306
epieeg60,Abnormal_Patient_59.EEG,1,0.89755994
epieeg60,Abnormal_Patient_60.EEG,1,0.87851554
epieeg60,Abnormal_Patient_61.EEG,1,0.56974614
epieeg60,Normal_Patient_01.EEG,0,0.68256974
epieeg60,Normal_Patient_02.EEG,0,0.59850556
epieeg60,Normal_Patient_03.EEG,0,0.46389803
epieeg60,Normal_Patient_04.EEG,0,0.40885425
epieeg60,Normal_Patient_05.EEG,0,0.36005417
epieeg60,Normal_Patient_06.EEG,0,0.6418556
epieeg60,Normal_Patient_07.EEG,0,0.4994891
epieeg60,Normal_Patient_08.EEG,0,0.5134616
epieeg60,Normal_Patient_09.EEG,0,0.8275915
epieeg60,Normal_Patient_10.EEG,0,0.40749836
epieeg60,Normal_Patient_11.EEG,0,0.5531171
epieeg60,Normal_Patient_12.EEG,0,0.6264398
epieeg60,Normal_Patient_13.EEG,0,0.4376668
epieeg60,Normal_Patient_14.EEG,0,0.86409193
epieeg60,Normal_Patient_15.EEG,0,0.57922053
epieeg60,Normal_Patient_16.EEG,0,0.87066025
epieeg60,Normal_Patient_17.EEG,0,0.6359175
epieeg60,Normal_Patient_18.EEG,0,0.5636945
epieeg60,Normal_Patient_19.EEG,0,0.35928
epieeg60,Normal_Patient_20.EEG,0,0.5697762
epieeg60,Normal_Patient_21.EEG,0,0.8750039
epieeg60,Normal_Patient_22.EEG,0,0.85347795
epieeg60,Normal_Patient_23.EEG,0,0.48554492
epieeg60,Normal_Patient_24.EEG,0,0.55273634
epieeg60,Normal_Patient_25.EEG,0,0.86930513
epieeg60,Normal_Patient_26.EEG,0,0.95564395
epieeg60,Normal_Patient_27.EEG,0,0.25761566
epieeg60,Normal_Patient_28.EEG,0,0.585666
epieeg60,Normal_Patient_29.EEG,0,0.50750816
epieeg60,Normal_Patient_30.EEG,0,0.42791703
epieeg60,Normal_Patient_N1.EEG,0,0.91948605
epieeg60,Normal_Patient_N2.EEG,0,0.73848027
"""
REVE = pd.read_csv(io.StringIO(REVE_CSV))

In [ ]:
ROOT = find_root()
meta, info = load_meta(ROOT)
print("общий набор:", ROOT, {k: info.get(k) for k in ["fs", "nmt_scale_k", "size_gb"]})
need = meta[meta.split.isin(["fit", "val", "eval", "turkey"])].copy()
if SMOKE:
    pick = lambda s, n0, n1: pd.concat([need[(need.split == s) & (need.y == 0)].head(n0), need[(need.split == s) & (need.y == 1)].head(n1)])
    need = pd.concat([pick("fit", 8, 4), pick("val", 3, 3), pick("eval", 3, 3), pick("turkey", 4, 4)])
REC = read_all(ROOT, need)
PREP = os.path.dirname(glob.glob("/kaggle/input/**/epieeg60_records.csv", recursive=True)[0])
ER = pd.read_csv(f"{PREP}/epieeg60_records.csv")
ER = ER[ER.n200.notna()].sort_values(["person", "id"]).reset_index(drop=True)
if SMOKE:
    ER = pd.concat([ER[ER.y == 0].head(6), ER[ER.y == 1].head(6)]).sort_values(["person", "id"]).reset_index(drop=True)
for r in ER.itertuples():
    REC[f"epi:{r.id}"] = np.load(f"{PREP}/fs200/{r.id}.npy")
assert all(v.shape[0] == 19 for v in REC.values())

ENT = []   # человек: набор, id, y, ключи записей
nm = {"fit": "nmt_fit", "val": "nmt_val", "eval": "nmt_eval", "turkey": "turkey"}
for r in need.itertuples():
    ENT.append((nm[r.split], r.id, int(r.y), [r.id]))
for p, g in ER.groupby("person", sort=True):
    ENT.append(("epieeg60", p, int(g.y.iloc[0]), [f"epi:{i}" for i in g.id]))
ED = pd.DataFrame([(a, b, c) for a, b, c, _ in ENT], columns=["dataset", "id", "y"])
KEYS = [k for *_, k in ENT]
print(pd.crosstab(ED.dataset, ED.y))
if not SMOKE:
    assert ED.groupby("dataset").size().to_dict() == {"nmt_fit": 1894, "nmt_val": 335, "nmt_eval": 185, "turkey": 121, "epieeg60": 63}
print("длины записей, с (медиана по набору):", {d: float(np.median([REC[k[0]].shape[1] / 200 for (dd, _, _, k) in ENT if dd == d])) for d in ED.dataset.unique()})
print(f"прочитано {len(REC)} записей | {time.time()-T00:.0f} с")

In [ ]:
BANDS = [(1, 4), (4, 8), (8, 13), (13, 20), (20, 30)]
BL = ["1-4", "4-8", "8-13", "13-20", "20-30"]
SOS = [ss.butter(4, b, btype="bandpass", fs=200, output="sos") for b in BANDS]
IU = np.triu_indices(19, 1)
PAIRS = [f"{EL[i]}-{EL[j]}" for i, j in zip(*IU)]
L, K, NB = 2000, 60, len(BANDS)

def pick_windows(lens):
    c = [(r, s) for r, n in enumerate(lens) for s in range(0, n - L + 1, L)]
    if len(c) > K:
        c = [c[i] for i in np.linspace(0, len(c) - 1, K).round().astype(int)]
    return c

def feats(e):
    arrs = [REC[k] for k in KEYS[e]]
    c = pick_windows([a.shape[1] for a in arrs])
    byrec = {}
    for r, s in c: byrec.setdefault(r, []).append(s)
    corr = np.zeros((NB, 171)); wp = np.zeros((NB, 171)); pw = np.zeros((NB, 19)); n = 0
    for r, starts in byrec.items():
        F = [ss.sosfiltfilt(s, np.asarray(arrs[r], np.float32), axis=-1) for s in SOS]
        for st in starts:
            v = np.empty((NB, 19))
            for b in range(NB):
                xb = F[b][:, st:st + L]
                v[b] = xb.var(-1)
                corr[b] += np.nan_to_num(np.corrcoef(xb)[IU])
                z = ss.hilbert(xb, axis=-1)
                im = (z[IU[0]] * np.conj(z[IU[1]])).imag
                wp[b] += np.abs(im.mean(-1)) / (np.abs(im).mean(-1) + 1e-12)
            pw += v / (v.sum(0, keepdims=True) + 1e-12)
            n += 1
    std = float(np.mean([np.asarray(a, np.float32).std() for a in arrs]))
    n = max(n, 1)
    return corr.ravel() / n, wp.ravel() / n, pw.ravel() / n, std, n

t0 = time.time()
with Pool(4) as pool:
    res = pool.map(feats, range(len(ENT)), chunksize=8)
FC = np.stack([r[0] for r in res]); FW = np.stack([r[1] for r in res]); FP = np.stack([r[2] for r in res])
ED["std_uv"] = [r[3] for r in res]; ED["n_win"] = [r[4] for r in res]
print(f"признаки: {time.time()-t0:.0f} с | окон на человека: мин {ED.n_win.min()}, медиана {ED.n_win.median():.0f}")
print(ED.groupby("dataset").n_win.describe()[["min", "50%", "max"]])
NAMES_C = [f"{p} {b} Гц" for b in BL for p in PAIRS]
NAMES_P = [f"{c} {b} Гц" for b in BL for c in EL]
np.savez_compressed(f"{OUT}/connectivity_features.npz", corr=FC.astype(np.float32), wpli=FW.astype(np.float32), power=FP.astype(np.float32), ids=ED.id.values, dataset=ED.dataset.values, y=ED.y.values)
SETS = {"power": (FP, NAMES_P), "corr": (FC, NAMES_C), "wpli": (FW, NAMES_C), "wpli_power": (np.hstack([FW, FP]), NAMES_C + [n + " (мощн.)" for n in NAMES_P])}
SETLAB = {"power": "мощность", "corr": "корреляция", "wpli": "wPLI", "wpli_power": "wPLI + мощность"}

In [ ]:
m = {d: (ED.dataset == d).values for d in ED.dataset.unique()}
y = ED.y.values
LR_C = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 0.1, 0.3, 1.0]
HGB_G = [dict(learning_rate=0.05, max_iter=150, max_depth=3), dict(learning_rate=0.05, max_iter=300, max_depth=3), dict(learning_rate=0.1, max_iter=100, max_leaf_nodes=15)]
def mk_lr(C): return make_pipeline(StandardScaler(), LogisticRegression(C=C, penalty="l2", class_weight="balanced", max_iter=3000))
def mk_hgb(g): return HistGradientBoostingClassifier(l2_regularization=1.0, class_weight="balanced", random_state=0, **g)

SEL, SCORE, MODELS = [], {}, {}
for sn, (X, _) in SETS.items():
    for mn, cands in [("lr", LR_C), ("hgb", HGB_G)]:
        best = None
        for p in cands:
            mdl = (mk_lr(p) if mn == "lr" else mk_hgb(p)).fit(X[m["nmt_fit"]], y[m["nmt_fit"]])
            s = mdl.predict_proba(X)[:, 1]
            a = roc_auc_score(y[m["nmt_val"]], s[m["nmt_val"]])
            SEL.append(dict(набор=sn, модель=mn, параметры=str(p), val_auc=round(a, 4)))
            if best is None or a > best[0]: best = (a, p, s, mdl)
        SCORE[(sn, mn)] = best[2]; MODELS[(sn, mn)] = (best[1], best[3])
        print(f"{sn:11s} {mn:4s} val AUC {best[0]:.3f} параметры {best[1]} | {time.time()-T00:.0f} с")
SEL = pd.DataFrame(SEL); SEL.to_csv(f"{OUT}/connectivity_model_selection.csv", index=False)
MAIN = {}   # лучшая модель для набора по val AUC
for sn in SETS:
    a = {mn: roc_auc_score(y[m["nmt_val"]], SCORE[(sn, mn)][m["nmt_val"]]) for mn in ["lr", "hgb"]}
    MAIN[sn] = max(a, key=a.get); print(sn, "основная модель:", MAIN[sn], {k: round(v, 3) for k, v in a.items()})

In [ ]:
# оценки REVE v4 и ансамбль (среднее рангов относительно val)
ED["key"] = ED.dataset + "|" + ED.id
RV = REVE.set_index(REVE.dataset + "|" + REVE.id)
fitmask = (ED.dataset == "nmt_fit").values
assert SMOKE or ED.key[~fitmask].isin(RV.index).all(), "нет оценок REVE для части людей"
reve = ED.key.map(RV.reve4_ens_prob).values
assert (ED.key.map(RV.y).fillna(ED.y).values == y).all()
S = {sn: SCORE[(sn, MAIN[sn])] for sn in SETS}
S["reve4"] = reve
CONN = ["corr", "wpli", "wpli_power"]
yv = y[m["nmt_val"]]
BEST = max(CONN, key=lambda sn: roc_auc_score(yv, S[sn][m["nmt_val"]]))
print("лучшая связность по val:", BEST)
def ecdf(s_ref, s): return np.searchsorted(np.sort(s_ref), s, side="right") / len(s_ref)
S["ens"] = (ecdf(S["reve4"][m["nmt_val"]], S["reve4"]) + ecdf(S[BEST][m["nmt_val"]], S[BEST])) / 2
LAB = {**SETLAB, "reve4": "REVE v4", "ens": f"REVE v4 + {SETLAB[BEST]} (среднее рангов)"}
TEST = ["nmt_eval", "turkey", "epieeg60"]
ALLK = ["reve4", "power", "corr", "wpli", "wpli_power", "ens"]
B = 2000

def boot_idx(n, B=B, seed=0):
    return np.random.default_rng(seed).integers(0, n, (B, n))
def auc_ci(yy, s_):
    a = roc_auc_score(yy, s_); v = []
    for i in boot_idx(len(yy)):
        if len(set(yy[i])) == 2: v.append(roc_auc_score(yy[i], s_[i]))
    return a, np.percentile(v, [2.5, 97.5])
fmt = lambda a, ci: f"{a:.3f} ({ci[0]:.3f}–{ci[1]:.3f})"
rows = []
for k in ALLK:
    row = {"признаки": LAB[k], "модель": MAIN.get(k, "—"), "val": round(roc_auc_score(yv, S[k][m["nmt_val"]]), 3)}
    for d in TEST:
        a, ci = auc_ci(y[m[d]], S[k][m[d]]); row[d] = fmt(a, ci)
    rows.append(row)
AUCT = pd.DataFrame(rows); AUCT.to_csv(f"{OUT}/connectivity_auc.csv", index=False)
pd.set_option("display.width", 250, "display.max_colwidth", 80)
print(AUCT.to_string(index=False))

# парная разница ROC AUC (бутстреп по людям)
PR = []
for a, b in [("ens", "reve4"), (BEST, "reve4"), ("power", "reve4"), ("corr", "reve4"), ("wpli", "reve4"), ("wpli_power", "reve4"), ("ens", BEST)]:
    for d in TEST:
        yy, s1, s2 = y[m[d]], S[a][m[d]], S[b][m[d]]; v = []
        for i in boot_idx(len(yy), seed=1):
            if len(set(yy[i])) == 2: v.append(roc_auc_score(yy[i], s1[i]) - roc_auc_score(yy[i], s2[i]))
        PR.append(dict(a=LAB[a], b=LAB[b], набор=d, разница=round(roc_auc_score(yy, s1) - roc_auc_score(yy, s2), 3), ДИ=f"{np.percentile(v,2.5):.3f}…{np.percentile(v,97.5):.3f}"))
PR = pd.DataFrame(PR); PR.to_csv(f"{OUT}/connectivity_paired.csv", index=False); print(PR.to_string(index=False))

In [ ]:
# пороговые метрики: порог по val (Юден и recall 80 %), переносится без изменений
def mets(yy, p):
    tp = ((p == 1) & (yy == 1)).sum(); fp = ((p == 1) & (yy == 0)).sum(); fn = ((p == 0) & (yy == 1)).sum(); tn = ((p == 0) & (yy == 0)).sum()
    rec = tp / max(tp + fn, 1); spec = tn / max(tn + fp, 1); prec = tp / (tp + fp) if tp + fp else np.nan
    return dict(accuracy=(tp + tn) / len(yy), bal_acc=(rec + spec) / 2, precision=prec, recall=rec, specificity=spec)
def thr_youden(yy, s): f, t, th = roc_curve(yy, s); return th[np.argmax(t - f)]
def thr_rec(yy, s, target=0.8): f, t, th = roc_curve(yy, s); return th[np.where(t >= target)[0][0]]
RULES = [("Юден", thr_youden), ("recall 80 %", thr_rec)]
TM = []
for k in ALLK:
    for rn, rf in RULES:
        th = rf(yv, S[k][m["nmt_val"]])
        for d in TEST:
            yy, s = y[m[d]], S[k][m[d]]; pt = mets(yy, (s >= th).astype(int))
            bs = pd.DataFrame([mets(yy[i], (s[i] >= th).astype(int)) for i in boot_idx(len(yy), B=1000, seed=2)])
            TM.append(dict(признаки=LAB[k], порог=rn, значение=round(float(th), 3), набор=d, **{c: f"{pt[c]:.2f} ({bs[c].quantile(.025):.2f}–{bs[c].quantile(.975):.2f})" for c in pt}))
TM = pd.DataFrame(TM); TM.to_csv(f"{OUT}/connectivity_threshold_metrics.csv", index=False)
print(TM[["признаки", "порог", "набор", "bal_acc", "recall", "specificity"]].to_string(index=False))

# калибровка порога на 20 людях новой больницы (10 + 10), 500 повторов, метрики на остальных
CAL = []
for d in ["turkey", "epieeg60"]:
    yy = y[m[d]]; i0, i1 = np.flatnonzero(yy == 0), np.flatnonzero(yy == 1); r = np.random.default_rng(0)
    draws = [np.concatenate([r.choice(i0, min(10, len(i0) - 1), replace=False), r.choice(i1, min(10, len(i1) - 1), replace=False)]) for _ in range(500)]
    for k in ALLK:
        s = S[k][m[d]]
        for rn, rf in RULES:
            R = []
            for cal in draws:
                test = np.setdiff1d(np.arange(len(yy)), cal); th = rf(yy[cal], s[cal]); R.append(mets(yy[test], (s[test] >= th).astype(int)))
            R = pd.DataFrame(R)
            CAL.append(dict(набор=d, признаки=LAB[k], правило=rn, **{c: f"{R[c].mean():.2f} ({R[c].quantile(.025):.2f}–{R[c].quantile(.975):.2f})" for c in ["bal_acc", "recall", "specificity", "precision", "accuracy"]}))
CAL = pd.DataFrame(CAL); CAL.to_csv(f"{OUT}/connectivity_calibration.csv", index=False)
print(CAL[["набор", "признаки", "правило", "bal_acc", "recall", "specificity"]].to_string(index=False))

In [ ]:
# проверки: опора на амплитуду (Спирмен с СКО записи внутри класса), возраст, главные признаки
def sp_ci(a, b, B_=1000):
    r0 = stats.spearmanr(a, b)[0]; v = [stats.spearmanr(a[i], b[i])[0] for i in boot_idx(len(a), B=B_, seed=3)]
    return r0, np.nanpercentile(v, [2.5, 97.5])
CK = []
for d in TEST:
    for cls, cn in [(0, "норма"), (1, "патология")]:
        sel = m[d] & (y == cls)
        for k in ALLK:
            r0, ci = sp_ci(S[k][sel], ED.std_uv.values[sel])
            CK.append(dict(проверка="Спирмен оценка–СКО", набор=d, класс=cn, признаки=LAB[k], n=int(sel.sum()), значение=round(r0, 2), ДИ=f"{ci[0]:.2f}…{ci[1]:.2f}"))
lab = sorted(glob.glob("/kaggle/input/**/[Ll]abels.csv", recursive=True))
if lab:
    LB = pd.read_csv(lab[0]).drop_duplicates("recordname").set_index("recordname")
    ED["age"] = ED.id.map(LB.age)
    sel0 = m["nmt_val"] | m["nmt_eval"]
    for cls, cn in [(0, "норма"), (1, "патология")]:
        sel = sel0 & (y == cls) & ED.age.notna().values
        for k in ALLK:
            r0, ci = sp_ci(S[k][sel], ED.age.values[sel])
            CK.append(dict(проверка="Спирмен оценка–возраст", набор="NMT val+eval", класс=cn, признаки=LAB[k], n=int(sel.sum()), значение=round(r0, 2), ДИ=f"{ci[0]:.2f}…{ci[1]:.2f}"))
    for k in ALLK:   # ложные тревоги среди детей и взрослых (порог Юден по val)
        th = thr_youden(yv, S[k][m["nmt_val"]])
        for gn, g in [("дети <18", ED.age < 18), ("взрослые >=18", ED.age >= 18)]:
            sel = sel0 & (y == 0) & g.values
            CK.append(dict(проверка="доля ложных тревог, порог Юден", набор="NMT val+eval", класс=gn, признаки=LAB[k], n=int(sel.sum()), значение=round(float((S[k][sel] >= th).mean()), 2), ДИ=""))
else:
    print("Labels.csv не найден — проверка возраста пропущена")
CK = pd.DataFrame(CK); CK.to_csv(f"{OUT}/connectivity_checks.csv", index=False)
print(CK[CK.проверка == "Спирмен оценка–СКО"].pivot_table(index=["набор", "класс"], columns="признаки", values="значение").to_string())
print(CK[CK.проверка != "Спирмен оценка–СКО"].to_string(index=False))

# главные признаки: стандартизованные коэффициенты логистической регрессии, обученной на NMT fit
TOP = []
for sn in ["corr", "wpli", "wpli_power"]:
    C_ = MODELS[(sn, "lr")][0]; pipe = mk_lr(C_).fit(SETS[sn][0][m["nmt_fit"]], y[m["nmt_fit"]]); co = pipe[-1].coef_[0]
    nms = SETS[sn][1]; o = np.argsort(-np.abs(co))
    for r_, i in enumerate(o[:25]): TOP.append(dict(набор=sn, ранг=r_ + 1, признак=nms[i], коэффициент=round(float(co[i]), 4), знак="выше при патологии" if co[i] > 0 else "ниже при патологии"))
    bl = pd.Series(np.abs(co[:NB * 171]).reshape(NB, 171).sum(1), index=BL); bl = bl / bl.sum()
    for b_, v in bl.items(): TOP.append(dict(набор=sn, ранг=0, признак=f"вклад диапазона {b_} Гц", коэффициент=round(float(v), 3), знак="доля суммы |коэфф.|"))
TOP = pd.DataFrame(TOP); TOP.to_csv(f"{OUT}/connectivity_top_features.csv", index=False)
print(TOP[TOP.набор == BEST].head(40).to_string(index=False))

# csv с оценками по людям (без fit — он обучающий)
mk = ~fitmask
PP = ED[mk][["dataset", "id", "y", "std_uv", "n_win"]].copy()
if "age" in ED: PP["age"] = ED.age.values[mk]
PP["reve4_ens_prob"] = S["reve4"][mk]
for sn in SETS:
    for mn in ["lr", "hgb"]: PP[f"{sn}_{mn}_prob"] = SCORE[(sn, mn)][mk]
    PP[f"{sn}_main_prob"] = S[sn][mk]
PP["ens_reve4_best_conn"] = S["ens"][mk]; PP["best_conn"] = BEST
PP.to_csv(f"{OUT}/connectivity_per_person.csv", index=False)
print("готово, всего", f"{time.time()-T00:.0f} с")

/usr/local/lib/python3.13/dist-packages/nbformat/validator.py:434: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  _validate(nbdict, ref, version, version_minor, relax_add_props)
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed.